### Q11 visualization: normalized within-trial firing rates for all mice

Trials have different durations, so every trial is warped piecewise into three normalized phases:

1. `trial_start` to `stim_start`
2. `stim_start` to `outcome`
3. `outcome` to `trial_end`

Each phase receives the same number of bins. This aligns trial start, stimulus, outcome, and trial end across all trials. Firing rates are still calculated using each bin's true duration in seconds, so the y-axis remains in Hz. The normalized x-axis represents trial phase rather than physical time.

Within each mouse and brain area, firing rates are first averaged equally across recorded units for each trial and then averaged across trials. One figure is saved per mouse. Each figure contains one panel per brain area, with separate colored trajectories for A=0 and A=1. Shaded bands show the trial-level SEM.

In [1]:
import csv
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

#### Parameters

In [2]:
project_folder = Path("E:/con2phys_submission")
repository_folder = Path("D:/git_repos/con2phys_submission")
sua_csv_path = project_folder / "dataset" / "full_sua_dataset.csv"
trials_csv_path = repository_folder / "results" / "q11" / "trials_info_filtered.csv"
q_path = repository_folder / "results" / "q11"
per_mouse_plot_folder = q_path / "per_mouse_plots"

EXPECTED_N_MICE = 18
BRAIN_AREAS = (1, 2, 3)
N_BINS_PER_PHASE = 20
N_PHASES = 3
N_NORMALIZED_BINS = N_BINS_PER_PHASE * N_PHASES

rates_csv_path = q_path / "all_mice_normalized_trial_firing_rates.csv"

#### Load and validate filtered trials

In [3]:
required_paths = [sua_csv_path, trials_csv_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required paths not found: {missing_paths}")

trials_df = pd.read_csv(trials_csv_path)
expected_trial_columns = [
    "mouse_id", "trial_id", "variable_A", "variable_B", "variable_C",
    "trial_start", "stim_start", "outcome", "trial_end", "trial_duration",
]
if list(trials_df.columns) != expected_trial_columns:
    raise ValueError(f"Unexpected trial schema: {list(trials_df.columns)}")
if trials_df[[
    "trial_start", "stim_start", "outcome", "trial_end"
]].isna().any().any():
    raise ValueError("Filtered trial table contains missing timing values")
if trials_df[["mouse_id", "trial_id"]].duplicated().any():
    raise ValueError("Duplicate mouse/trial identifiers found")

correct_order = (
    trials_df["trial_start"].lt(trials_df["stim_start"])
    & trials_df["stim_start"].lt(trials_df["outcome"])
    & trials_df["outcome"].lt(trials_df["trial_end"])
)
if not correct_order.all():
    raise ValueError("Filtered trial table contains incorrectly ordered events")
if set(trials_df["variable_A"].unique()) != {0, 1}:
    raise ValueError("variable_A must contain both binary classes")

mouse_ids = sorted(int(mouse_id) for mouse_id in trials_df["mouse_id"].unique())
if len(mouse_ids) != EXPECTED_N_MICE:
    raise ValueError(f"Expected {EXPECTED_N_MICE} mice, found {len(mouse_ids)}")

trials_by_mouse = {
    int(mouse_id): group.sort_values("trial_id").reset_index(drop=True)
    for mouse_id, group in trials_df.groupby("mouse_id", sort=True)
}
class_counts = pd.crosstab(trials_df["mouse_id"], trials_df["variable_A"])
if (class_counts == 0).any().any():
    raise ValueError("Every mouse must contain both values of variable_A")
display(class_counts.rename(columns={0: "A=0", 1: "A=1"}))
print(f"Validated {len(trials_df):,} trials from {len(mouse_ids)} mice.")

variable_A,A=0,A=1
mouse_id,,
1,73,334
2,88,761
3,97,376
4,26,478
5,94,329
6,98,372
7,117,358
8,146,516
9,107,472


Validated 10,350 trials from 18 mice.


#### Construct trial-specific normalized bins

The same normalized bin has different physical widths in different trials. `bin_duration_sec` stores those true widths and is used when converting spike counts to Hz. No spikes from inter-trial intervals are included.

In [4]:
def piecewise_trial_edges(trial_row, n_bins_per_phase):
    event_times = np.array([
        trial_row["trial_start"],
        trial_row["stim_start"],
        trial_row["outcome"],
        trial_row["trial_end"],
    ], dtype=float)
    phase_edges = []
    for phase_index in range(3):
        edges = np.linspace(
            event_times[phase_index],
            event_times[phase_index + 1],
            n_bins_per_phase + 1,
        )
        if phase_index:
            edges = edges[1:]
        phase_edges.append(edges)
    return np.concatenate(phase_edges)


trial_edges_by_mouse = {}
bin_durations_by_mouse = {}
for mouse_id, mouse_trials in trials_by_mouse.items():
    edges = np.vstack([
        piecewise_trial_edges(row, N_BINS_PER_PHASE)
        for _, row in mouse_trials.iterrows()
    ])
    durations = np.diff(edges, axis=1)
    expected_shape = (len(mouse_trials), N_NORMALIZED_BINS + 1)
    if edges.shape != expected_shape:
        raise ValueError(f"Unexpected normalized edge shape for mouse {mouse_id}")
    if not np.isfinite(durations).all() or np.any(durations <= 0):
        raise ValueError(f"Invalid normalized durations for mouse {mouse_id}")
    trial_edges_by_mouse[mouse_id] = edges
    bin_durations_by_mouse[mouse_id] = durations

normalized_centers = (
    np.arange(N_NORMALIZED_BINS, dtype=float) + 0.5
) / N_BINS_PER_PHASE
phase_names = np.repeat([
    "trial_start_to_stim", "stim_to_outcome", "outcome_to_trial_end"
], N_BINS_PER_PHASE)
phase_fractions = np.tile(
    (np.arange(N_BINS_PER_PHASE) + 0.5) / N_BINS_PER_PHASE,
    N_PHASES,
)

portion_duration_summary = (
    trials_df.assign(
        start_to_stim=trials_df["stim_start"] - trials_df["trial_start"],
        stim_to_outcome=trials_df["outcome"] - trials_df["stim_start"],
        outcome_to_end=trials_df["trial_end"] - trials_df["outcome"],
    )
    .groupby("mouse_id")[[
        "start_to_stim", "stim_to_outcome", "outcome_to_end"
    ]]
    .median()
)
display(portion_duration_summary)

,start_to_stim,stim_to_outcome,outcome_to_end
mouse_id,,,
1,0.710429,0.406526,1.109922
2,0.697667,0.378300,1.066467
3,0.647506,0.634174,1.064434
4,1.887728,0.280955,1.058144
5,0.738026,0.603173,1.111422
6,0.606193,0.298855,1.066200
7,0.612145,0.841224,1.058604
8,0.614612,0.391815,1.062272
9,0.637842,0.400731,1.060637


#### Calculate population firing rates

For every unit and trial, spikes are counted in the trial-specific normalized bins. Each count is divided by its true bin duration. Unit rates are then averaged within brain area, giving one population-mean firing-rate trajectory per trial and area.

In [5]:
expected_sua_columns = [
    "mouse_id", "cluster_id", "brain_area", "recording_duration_sec",
    "unit_last_spike_sec", "n_spikes", "spikes",
]
area_rate_sums = {
    (mouse_id, brain_area): np.zeros(
        (len(trials_by_mouse[mouse_id]), N_NORMALIZED_BINS), dtype=float
    )
    for mouse_id in mouse_ids
    for brain_area in BRAIN_AREAS
}
unit_counts = defaultdict(int)
unit_keys = set()

csv.field_size_limit(2**31 - 1)
with sua_csv_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if reader.fieldnames != expected_sua_columns:
        raise ValueError(f"Unexpected SUA schema: {reader.fieldnames}")

    for row in reader:
        mouse_id = int(row["mouse_id"])
        if mouse_id not in trials_by_mouse:
            continue

        cluster_id = int(row["cluster_id"])
        brain_area = int(row["brain_area"])
        unit_key = (mouse_id, cluster_id)
        group_key = (mouse_id, brain_area)
        if unit_key in unit_keys:
            raise ValueError(f"Duplicate unit key: {unit_key}")
        if brain_area not in BRAIN_AREAS:
            raise ValueError(f"Unexpected brain area for {unit_key}: {brain_area}")

        spike_times = np.fromstring(row["spikes"], sep=";", dtype=np.float64)
        expected_n_spikes = int(row["n_spikes"])
        if spike_times.size != expected_n_spikes or spike_times.size == 0:
            raise ValueError(f"Spike-count mismatch for {unit_key}")
        if not np.isfinite(spike_times).all() or np.any(np.diff(spike_times) < 0):
            raise ValueError(f"Invalid or unsorted spikes for {unit_key}")

        edges = trial_edges_by_mouse[mouse_id]
        durations = bin_durations_by_mouse[mouse_id]
        edge_indices = np.searchsorted(
            spike_times, edges.reshape(-1), side="left"
        ).reshape(edges.shape)
        unit_rates = np.diff(edge_indices, axis=1) / durations

        area_rate_sums[group_key] += unit_rates
        unit_counts[group_key] += 1
        unit_keys.add(unit_key)

expected_group_keys = {
    (mouse_id, brain_area)
    for mouse_id in mouse_ids
    for brain_area in BRAIN_AREAS
}
if set(unit_counts) != expected_group_keys:
    raise ValueError(
        f"Missing mouse/area groups: {sorted(expected_group_keys - set(unit_counts))}"
    )

population_rates = {
    group_key: area_rate_sums[group_key] / unit_counts[group_key]
    for group_key in expected_group_keys
}
if not all(np.isfinite(values).all() for values in population_rates.values()):
    raise ValueError("Population firing rates contain non-finite values")

unit_counts_df = pd.DataFrame([
    {
        "mouse_id": mouse_id,
        "brain_area": brain_area,
        "n_units": unit_counts[(mouse_id, brain_area)],
    }
    for mouse_id, brain_area in sorted(expected_group_keys)
])
display(unit_counts_df)

,mouse_id,brain_area,n_units
0,1,1,59
1,1,2,22
2,1,3,7
3,2,1,40
4,2,2,20
5,2,3,19
6,3,1,47
7,3,2,26
8,3,3,12
9,4,1,34


#### Summarize and compare A=0 with A=1

In [6]:
summary_rows = []
class_colors = {0: "tab:blue", 1: "tab:orange"}
event_positions = [0, 1, 2, 3]
event_labels = ["Trial start", "Stimulus", "Outcome", "Trial end"]
per_mouse_plot_folder.mkdir(parents=True, exist_ok=True)

for mouse_id in mouse_ids:
    mouse_trials = trials_by_mouse[mouse_id]
    labels = mouse_trials["variable_A"].to_numpy(dtype=int)
    class_summaries = {}

    for variable_a in (0, 1):
        class_mask = labels == variable_a
        class_summaries[variable_a] = {}
        for brain_area in BRAIN_AREAS:
            group_key = (mouse_id, brain_area)
            trial_rates = population_rates[group_key][class_mask]
            mean_rate = trial_rates.mean(axis=0)
            sem_rate = (
                trial_rates.std(axis=0, ddof=1) / np.sqrt(trial_rates.shape[0])
            )
            class_summaries[variable_a][brain_area] = (mean_rate, sem_rate)

            for bin_index in range(N_NORMALIZED_BINS):
                summary_rows.append({
                    "mouse_id": mouse_id,
                    "brain_area": brain_area,
                    "variable_A": variable_a,
                    "normalized_time": normalized_centers[bin_index],
                    "phase": phase_names[bin_index],
                    "phase_fraction": phase_fractions[bin_index],
                    "mean_firing_rate_hz": mean_rate[bin_index],
                    "sem_across_trials_hz": sem_rate[bin_index],
                    "n_trials": int(class_mask.sum()),
                    "n_units": unit_counts[group_key],
                })

    upper_values = [
        mean_rate + sem_rate
        for class_data in class_summaries.values()
        for mean_rate, sem_rate in class_data.values()
    ]
    common_y_max = max(values.max() for values in upper_values) * 1.05

    fig, axes = plt.subplots(
        1, len(BRAIN_AREAS), figsize=(13, 3.8), sharex=True, sharey=True
    )
    for axis, brain_area in zip(axes, BRAIN_AREAS):
        for variable_a in (0, 1):
            mean_rate, sem_rate = class_summaries[variable_a][brain_area]
            color = class_colors[variable_a]
            axis.plot(
                normalized_centers, mean_rate, color=color, linewidth=1.8,
                label=f"A={variable_a} (n={int((labels == variable_a).sum())})",
            )
            axis.fill_between(
                normalized_centers,
                mean_rate - sem_rate,
                mean_rate + sem_rate,
                color=color,
                alpha=0.22,
                linewidth=0,
            )
        for event_position in event_positions[1:-1]:
            axis.axvline(
                event_position, color="0.35", linestyle="--", linewidth=0.9
            )
        axis.set_title(
            f"Brain area {brain_area} "
            f"(n={unit_counts[(mouse_id, brain_area)]} units)"
        )
        axis.set_xticks(event_positions, event_labels, rotation=20)
        axis.set_xlim(0, 3)
        axis.set_ylim(0, common_y_max)
        axis.set_xlabel("Normalized trial phase")
        axis.grid(True, axis="y", alpha=0.2)
        axis.legend(frameon=False, loc="best")
        sns.despine(ax=axis)

    axes[0].set_ylabel("Population mean firing rate (Hz)")
    fig.suptitle(
        f"Mouse {mouse_id}: normalized within-trial firing rate by variable A"
    )
    fig.tight_layout()
    figure_path = (
        per_mouse_plot_folder
        / f"mouse_{mouse_id:02d}_normalized_firing_rate_by_A.png"
    )
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved mouse {mouse_id} figure: {figure_path.name}")

normalized_rates_df = pd.DataFrame(summary_rows)
q_path.mkdir(parents=True, exist_ok=True)
normalized_rates_df.to_csv(rates_csv_path, index=False)

display(normalized_rates_df.head())
print(f"Saved {len(normalized_rates_df):,} normalized summary rows.")
print(f"Saved normalized data to {rates_csv_path}")
print(f"Saved {len(mouse_ids)} figures to {per_mouse_plot_folder}")

Saved mouse 1 figure: mouse_01_normalized_firing_rate_by_A.png


Saved mouse 2 figure: mouse_02_normalized_firing_rate_by_A.png


Saved mouse 3 figure: mouse_03_normalized_firing_rate_by_A.png


Saved mouse 4 figure: mouse_04_normalized_firing_rate_by_A.png


Saved mouse 5 figure: mouse_05_normalized_firing_rate_by_A.png


Saved mouse 6 figure: mouse_06_normalized_firing_rate_by_A.png


... (10 outputs left out)

Saved mouse 17 figure: mouse_17_normalized_firing_rate_by_A.png


Saved mouse 18 figure: mouse_18_normalized_firing_rate_by_A.png


,mouse_id,brain_area,variable_A,normalized_time,phase,phase_fraction,mean_firing_rate_hz,sem_across_trials_hz,n_trials,n_units
0,1,1,0,0.025,trial_start_to_stim,0.025,8.085117,0.302085,73,59
1,1,1,0,0.075,trial_start_to_stim,0.075,8.077329,0.356059,73,59
2,1,1,0,0.125,trial_start_to_stim,0.125,7.758723,0.349760,73,59
3,1,1,0,0.175,trial_start_to_stim,0.175,7.792431,0.363967,73,59
4,1,1,0,0.225,trial_start_to_stim,0.225,7.982347,0.400739,73,59


Saved 6,480 normalized summary rows.
Saved normalized data to D:\git_repos\con2phys_submission\results\q11\all_mice_normalized_trial_firing_rates.csv
Saved 18 figures to D:\git_repos\con2phys_submission\results\q11\per_mouse_plots
